# maniol-elvish ポーズ・画像変数インスペクション（Snapshot Inspector）

キャラクター `maniol-elvish` を対象に、立ち絵差分やアニメーション用フレームの一貫した画像変数定義・チェックを行うノートブックです。  
共通処理は [`../common/`](../common/) パッケージに分離しており、このノートブックでは **`maniol-elvish` 固有のチェック変数設計・しきい値調整・検証** に集中できます。

1. **セットアップ**: 必要パッケージの導入と GPU 確認
2. **Drive マウント & 共通モジュール読み込み**: リポジトリ取得と `common` パッケージのインポート・モデル初期化
3. **データ準備**: `reference_set`（複数 LoRA 学習データ）と `targets`（検査対象画像）の配置
4. **`maniol-elvish` チェック変数設計**: 可変スロット・固有特徴タグ・検査しきい値の定義
5. **キャラクター固有分布（Character Profile）の構築**
6. **対象画像の変数化 & 一貫性チェック・可視化**
7. **結果の Drive 保存・ランタイム終了メモ**

## 1. セットアップ

In [ ]:
%pip install -q rtmlib onnxruntime-gpu opencv-python huggingface_hub pandas matplotlib

In [ ]:
!nvidia-smi

## 2. Google Drive マウント & 共通モジュール読み込み

`comfyui/colab_runner.ipynb` と同様にリポジトリを `/content/stable-diffusion-pipelines` へ取得（または更新）し、`colab/snapshot-inspector/common` から共通関数を読み込みます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/belre/stable-diffusion-pipelines.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/stable-diffusion-pipelines")

# ローカル環境で直接開いた場合は親ディレクトリを優先し、Colab上ではリポジトリを clone/pull する
local_inspector_root = Path("..").resolve()
if (local_inspector_root / "common" / "__init__.py").exists():
    INSPECTOR_ROOT = local_inspector_root
else:
    if not REPO_DIR.exists():
        !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
    INSPECTOR_ROOT = REPO_DIR / "colab" / "snapshot-inspector"

if str(INSPECTOR_ROOT) not in sys.path:
    sys.path.insert(0, str(INSPECTOR_ROOT))

import os
import json
import copy
from datetime import datetime, timezone

import cv2
import pandas as pd

from common import (
    DEFAULT_IGNORED_IDENTITY_TAGS,
    DEFAULT_VARIABLE_TAG_SLOTS,
    DEFAULT_WD14_REPO,
    WD14Tagger,
    build_character_profile,
    evaluate_against_profile,
    extract_image_variables,
    get_onnx_device_config,
    init_wholebody,
    list_image_files,
    save_results_to_drive,
    show_batch_variable_heatmap,
    show_profile_summary,
    show_target_report,
    variables_to_flat_row,
)

rtmlib_device, ort_providers = get_onnx_device_config()
print(f"INSPECTOR_ROOT: {INSPECTOR_ROOT}")
print(f"ONNX device: {rtmlib_device} / providers: {ort_providers}")

# モデル初期化
wholebody = init_wholebody(mode="performance", device=rtmlib_device)
wd14_tagger = WD14Tagger(repo_id=DEFAULT_WD14_REPO, providers=ort_providers)

## 3. データ準備（`maniol-elvish`）

- `./data/maniol-elvish/reference_set/`: キャラクター固有分布を作るための複数 LoRA 学習データ／リファレンス画像群
- `./data/maniol-elvish/targets/`: 検査・変数化したい対象画像（立ち絵差分・アニメーションフレーム等）

In [ ]:
CHARACTER_NAME = "maniol-elvish"
DATA_ROOT = f"./data/{CHARACTER_NAME}"
REFERENCE_DIR = f"{DATA_ROOT}/reference_set"
TARGET_DIR = f"{DATA_ROOT}/targets"

!mkdir -p {REFERENCE_DIR} {TARGET_DIR}

# Google Drive からのコピー例（パスは環境に合わせてコメント解除・編集してください）
# !cp '/content/drive/MyDrive/Colab Notebooks/maniol-elvish/reference/'*.png {REFERENCE_DIR}/
# !cp '/content/drive/MyDrive/Colab Notebooks/maniol-elvish/targets/'*.png {TARGET_DIR}/

!echo "=== reference_set ===" && ls -1 {REFERENCE_DIR} | head -n 20
!echo "=== targets ===" && ls -1 {TARGET_DIR} | head -n 20

## 4. `maniol-elvish` チェック変数の設計・パラメータ設定

キャラクター固有のチェック変数・スロット・しきい値をここで管理します。  
今後の設計に合わせて、スロットへのタグ追加や強制チェック対象（`FORCE_IDENTITY_TAGS`）を段階的に調整します。

In [ ]:
# ==========================================
# maniol-elvish 向けチェック変数設定
# ==========================================

# 1. 可変スロット定義（デフォルトをコピーしてキャラ固有に拡張可能）
VARIABLE_TAG_SLOTS = copy.deepcopy(DEFAULT_VARIABLE_TAG_SLOTS)
IGNORED_IDENTITY_TAGS = set(DEFAULT_IGNORED_IDENTITY_TAGS)

# 2. キャラ固有不変特徴の手動調整（必要に応じて追記）
FORCE_IDENTITY_TAGS = [
    # 例: "pointy_ears", "blonde_hair" など
]
EXCLUDE_IDENTITY_TAGS = [
    # 例: 衣装差分などで変動を許容したいタグ
]

# 3. 固有分布抽出・検査しきい値
CORE_PRESENCE_THRESHOLD = 0.65   # リファレンス画像のうち何割以上で検出されたタグを固有特徴とするか
CORE_MEAN_THRESHOLD = 0.45       # 平均信頼度がこの値以上のタグを固有特徴とするか
TAG_ACTIVE_THRESHOLD = 0.35      # タグ出現判定の信頼度しきい値
KPT_THR = 0.35                   # 骨格キーポイント有効しきい値

# 4. 一貫性チェックの警告感度
IDENTITY_Z_WARN = 2.0            # 固有特徴タグが平均から何σ以上低下したら警告するか
PROPORTION_Z_WARN = 2.5          # 骨格プロポーションが平均から何σ以上ズレたら警告するか
UNEXPECTED_TAG_THRESHOLD = 0.55  # 学習分布にない外乱タグの警告しきい値

# 5. 出力パス設定
OUTPUT_DIR = f"./output/{CHARACTER_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
PROFILE_JSON_PATH = f"{OUTPUT_DIR}/character_profile.json"
OUTPUT_JSON_PATH = f"{OUTPUT_DIR}/image_variables.json"
OUTPUT_CSV_PATH = f"{OUTPUT_DIR}/image_variables_summary.csv"
DEBUG_SKEL_DIR = f"{OUTPUT_DIR}/debug_skeletons"

# 既存プロファイルを再利用する場合はパスを指定（新規作成時は None）
LOAD_EXISTING_PROFILE_PATH = None

print(f"Configured inspection for '{CHARACTER_NAME}' -> output: {OUTPUT_DIR}")

## 5. キャラクター固有分布（Character Profile）の構築

In [ ]:
if LOAD_EXISTING_PROFILE_PATH and os.path.exists(LOAD_EXISTING_PROFILE_PATH):
    with open(LOAD_EXISTING_PROFILE_PATH, "r", encoding="utf-8") as f:
        character_profile = json.load(f)
    print(f"✅ 既存プロファイルを読み込みました: {LOAD_EXISTING_PROFILE_PATH}")
else:
    ref_paths = list_image_files(REFERENCE_DIR)
    print(f"リファレンス画像数: {len(ref_paths)} ({REFERENCE_DIR})")
    character_profile, ref_records = build_character_profile(
        reference_paths=ref_paths,
        pose_model=wholebody,
        tagger=wd14_tagger,
        character_name=CHARACTER_NAME,
        slot_definitions=VARIABLE_TAG_SLOTS,
        ignored_identity_tags=IGNORED_IDENTITY_TAGS,
        force_identity_tags=FORCE_IDENTITY_TAGS,
        exclude_identity_tags=EXCLUDE_IDENTITY_TAGS,
        core_presence_threshold=CORE_PRESENCE_THRESHOLD,
        core_mean_threshold=CORE_MEAN_THRESHOLD,
        tag_active_threshold=TAG_ACTIVE_THRESHOLD,
        kpt_thr=KPT_THR,
    )
    with open(PROFILE_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(character_profile, f, ensure_ascii=False, indent=2)
    print(f"✅ Character Profile を保存しました -> {PROFILE_JSON_PATH}")
    print(f"   抽出されたキャラ固有不変タグ数: {len(character_profile['identity_core_tags'])}")

show_profile_summary(character_profile, top_n=20)

## 6. 対象画像の変数化 & 一貫性チェック実行

In [ ]:
target_paths = list_image_files(TARGET_DIR)
assert len(target_paths) > 0, f"対象画像が見つかりません: {TARGET_DIR}"
os.makedirs(DEBUG_SKEL_DIR, exist_ok=True)

target_results = []
debug_skels = {}
summary_rows = []

for idx, path in enumerate(target_paths, 1):
    rec, wd14_pred, _, debug_skel = extract_image_variables(
        path,
        pose_model=wholebody,
        tagger=wd14_tagger,
        slot_definitions=VARIABLE_TAG_SLOTS,
        kpt_thr=KPT_THR,
        general_threshold=TAG_ACTIVE_THRESHOLD,
        return_debug_artifacts=True,
    )
    consistency = evaluate_against_profile(
        record=rec,
        wd14_pred=wd14_pred,
        profile=character_profile,
        slot_definitions=VARIABLE_TAG_SLOTS,
        ignored_identity_tags=IGNORED_IDENTITY_TAGS,
        identity_z_warn=IDENTITY_Z_WARN,
        proportion_z_warn=PROPORTION_Z_WARN,
        unexpected_tag_threshold=UNEXPECTED_TAG_THRESHOLD,
    )

    skel_path = os.path.join(DEBUG_SKEL_DIR, f"{rec['image_id']}_skeleton.png")
    cv2.imwrite(skel_path, debug_skel)
    debug_skels[rec["image_id"]] = debug_skel

    combined = {**rec, "consistency": consistency}
    target_results.append(combined)
    summary_rows.append(variables_to_flat_row(rec, consistency))
    print(
        f"[Target {idx}/{len(target_paths)}] {rec['file_name']} | "
        f"overall={consistency['overall_consistency']} "
        f"(id={consistency['identity_consistency']}, prop={consistency['proportion_consistency']})"
    )

payload = {
    "schema_version": "1.0",
    "character_name": character_profile.get("character_name"),
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "target_count": len(target_results),
    "images": target_results,
}
with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"\n✅ 変数定義 JSON 保存: {OUTPUT_JSON_PATH}")
print(f"✅ 一覧サマリー CSV 保存: {OUTPUT_CSV_PATH}")

display(summary_df)

In [ ]:
MAX_REPORTS_TO_SHOW = 10

for item in target_results[:MAX_REPORTS_TO_SHOW]:
    print(f"\n==================== {item['file_name']} ====================")
    show_target_report(item, item["consistency"], debug_skels[item["image_id"]])

show_batch_variable_heatmap(target_results, slot_name="expression", min_max_score=0.20)
show_batch_variable_heatmap(target_results, slot_name="gaze", min_max_score=0.20)

## 7. 結果のDrive保存・ランタイム終了メモ

In [ ]:
DRIVE_OUTPUT_DIR = f"/content/drive/MyDrive/Colab Notebooks/snapshot-inspector/{CHARACTER_NAME}"
OUTPUT_PREFIX = f"{CHARACTER_NAME}_"

TARGET_FILES = [
    PROFILE_JSON_PATH,
    OUTPUT_JSON_PATH,
    OUTPUT_CSV_PATH,
]

save_results_to_drive(DRIVE_OUTPUT_DIR, file_paths=TARGET_FILES, prefix=OUTPUT_PREFIX)

### VS Code 上でのランタイム終了手順
作業が終わったら、GPUランタイムの課金・リソース消費を防ぐため、必ずランタイムを切断・削除してください。

1. VS Code 上部のコマンドパレット（`Ctrl + Shift + P` / macOS は `Cmd + Shift + P`）を開く
2. **`Colab: Remove Server`** を入力して実行する
3. 確認ダイアログが出たら対象の Colab ランタイムを削除して終了する